In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model

ModuleNotFoundError: No module named 'sklearn'

# Lectura de CSV

In [ ]:
# Ruta de la carpeta donde están los CSV
carpeta_csv = "./data/clean"

# Obtener todos los archivos CSV de la carpeta
archivos = [f for f in os.listdir(carpeta_csv)] #if f.endswith('.csv')]

# Lista para almacenar cada DataFrame
lista_df = []

In [52]:
len(archivos)

223

In [54]:
# Leer cada CSV y agregarlo a la lista
for archivo in archivos:
    ruta_archivo = os.path.join(carpeta_csv, archivo)
    df = pd.read_csv(ruta_archivo)
    lista_df.append(df)

# Unir todos los DataFrames en uno solo
df_final = pd.concat(lista_df, ignore_index=True)

# Mostrar las primeras filas del DataFrame final
print(df_final.head())

    time  torque_spindle  velocity_spindle            source_file       path  \
0  0.000         1.82520       76207.50000  Trace_0622_172347.csv  C1030/V01   
1  0.002         1.69189       75843.64063  Trace_0622_172347.csv  C1030/V01   
2  0.004         1.58936       75862.61719  Trace_0622_172347.csv  C1030/V01   
3  0.006         1.48682       75963.86719  Trace_0622_172347.csv  C1030/V01   
4  0.008         1.66113       75841.52344  Trace_0622_172347.csv  C1030/V01   

   Ae_mm  Ap_mm  f_mm_min   N_rpm  
0   40.0    1.0     632.0  2108.0  
1   40.0    1.0     632.0  2108.0  
2   40.0    1.0     632.0  2108.0  
3   40.0    1.0     632.0  2108.0  
4   40.0    1.0     632.0  2108.0  


In [58]:
df_final.dtypes

time                float64
torque_spindle      float64
velocity_spindle    float64
source_file          object
path                 object
Ae_mm               float64
Ap_mm               float64
f_mm_min            float64
N_rpm               float64
dtype: object

In [66]:
df = df_final.drop(columns = ['path'])

# Preparación de los datos

In [70]:
columnas_features = ['velocity_spindle', 'Ae_mm', 'Ap_mm', 'f_mm_min', 'N_rpm']
df_features = df[['source_file', 'time'] + columnas_features]

# Normalizar los features
scaler = MinMaxScaler()
df[columnas_features] = scaler.fit_transform(df[columnas_features])

In [74]:
timesteps = 10

def crear_secuencias(df, timesteps, feature_cols, target_col='torque_spindle'):
    X_list = []
    y_list = []
    
    # Iterar por cada serie de tiempo
    for id_serie, group in df.groupby('source_file'):
        group = group.sort_values('time')
        features = group[feature_cols].values
        target = group[target_col].values
        
        for i in range(len(group) - timesteps):
            X_list.append(features[i:i+timesteps])
            y_list.append(target[i+timesteps])
    
    return np.array(X_list), np.array(y_list)

X, y = crear_secuencias(df, timesteps, columnas_features)
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)


Shape de X: (1853869, 10, 5)
Shape de y: (1853869,)


In [76]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# LTSM

In [78]:
#Construcción del modelo
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(timesteps, len(columnas_features))))
model.add(Dense(1))  # Predicción de y
model.compile(optimizer='adam', loss='mse')

model.summary()

C:\Users\rhuer\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        11,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,251 (43.95 KB)

 Trainable params: 11,251 (43.95 KB)

 Non-trainable params: 0 (0.00 B)

In [80]:
#Entrenamiento del modelo
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32
)


Epoch 1/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - loss: nan - val_loss: nan
Epoch 2/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - loss: nan - val_loss: nan
Epoch 3/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - loss: nan - val_loss: nan
Epoch 4/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - loss: nan - val_loss: nan
Epoch 5/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - loss: nan - val_loss: nan
Epoch 6/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - loss: nan - val_loss: nan
Epoch 7/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - loss: nan - val_loss: nan
Epoch 8/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 139s 3ms/step - loss: nan - val_loss: nan
Epoch 9/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 140s 3ms/step - loss: nan - val_loss: nan
Epoch 10/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - loss: nan - val_loss: nan
Epoch 11/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - loss: nan - val_loss: nan
Epoch 12/50
46347/46347 ━━━━━━━━━━━━━━━━━━━━ 140s 3m

In [82]:
model.save("mi_modelo_lstm.keras") #Guardar modelo

In [ ]:
model = load_model("mi_modelo_lstm.keras") #Cargar modelo